# 01 · Data understanding
Goal: know exactly what is in the data **before** any modelling: structure, types, missing values,
duplicates, the target, identifiers, timestamps and leakage risks. Data is loaded with the Hugging Face
`datasets` library (`src/data_loader.py`) and cached locally as Parquet.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
from src import config
from src.data_loader import load_splits
from src import inspect_data as insp
splits = load_splits()
{k: v.shape for k, v in splits.items()}

{'test': (555719, 24), 'train': (1296675, 24)}

## Columns, types, missing values, cardinality (train)

In [2]:
train, test = splits['train'], splits['test']
insp.column_profile(train)

,dtype,n_missing,pct_missing,n_unique,unique_ratio,example
Unnamed: 0,int64,0,0.000,1296675,1.0000,np.int64(0)
trans_date_trans_time,str,0,0.000,1274791,0.9831,'2019-01-01 00:00:18'
cc_num,int64,0,0.000,983,0.0008,np.int64(2703186189652095)
merchant,str,0,0.000,693,0.0005,"'fraud_Rippin, Kub and Mann'"
category,str,0,0.000,14,0.0000,'misc_net'
amt,float64,0,0.000,52928,0.0408,np.float64(4.97)
first,str,0,0.000,352,0.0003,'Jennifer'
last,str,0,0.000,481,0.0004,'Banks'
gender,str,0,0.000,2,0.0000,'F'
street,str,0,0.000,983,0.0008,'561 Perry Cove'


## Duplicates
Full-row duplicates, and duplicates after removing the row-index column.

In [3]:
{n: (int(df.duplicated().sum()), int(df.drop(columns=['Unnamed: 0']).duplicated().sum())) for n, df in splits.items()}

{'test': (0, 0), 'train': (0, 0)}

## Target and class imbalance

In [4]:
pd.DataFrame({n: df[config.TARGET].value_counts() for n, df in splits.items()}).assign(
    train_pct=lambda d: 100*d['train']/d['train'].sum(), test_pct=lambda d: 100*d['test']/d['test'].sum())

,test,train,train_pct,test_pct
is_fraud,,,,
0,553574,1289169,99.421135,99.614014
1,2145,7506,0.578865,0.385986


## Identifiers and timestamps
Candidate identifiers are flagged by name pattern or near-unique values; they need human judgement.

In [5]:
display(insp.candidate_identifiers(train, config.TARGET))
insp.detect_datetime_columns(train)

,column,unique_ratio,name_matches_id_pattern
0,Unnamed: 0,1.0000,True
1,trans_date_trans_time,0.9831,True
2,cc_num,0.0008,True
3,trans_num,1.0000,True
4,unix_time,0.9831,False
5,merch_lat,0.9623,False
6,merch_long,0.9839,False


{'trans_date_trans_time': 'string parses as datetime (range sample 2019-01-01 16:52:19 → 2020-06-21 06:41:38)',
 'dob': 'string parses as datetime (range sample 1924-10-30 00:00:00 → 2005-01-29 00:00:00)',
 'unix_time': 'integer in epoch-seconds range (2012-01-01 00:00:18 → 2013-06-21 12:13:37)'}

## Train/test relationship
Is the split by time? Do rows or IDs overlap?

In [6]:
tr_t, te_t = pd.to_datetime(train[config.TIME_COL]), pd.to_datetime(test[config.TIME_COL])
print('train', tr_t.min(), '->', tr_t.max()); print('test ', te_t.min(), '->', te_t.max())
insp.cross_split_overlap(splits, ['trans_num', 'cc_num'], config.TARGET)

train 2019-01-01 00:00:18 -> 2020-06-21 12:13:37
test  2020-06-21 12:14:25 -> 2020-12-31 23:59:34


,what,n_shared,pct_of_test_unique
0,values of `trans_num`,0,0.00
1,values of `cc_num`,908,98.27
2,exact rows (excluding index-like cols),0,0.00


## Card = customer?
Each card number maps to a single customer profile, so `cc_num` is the natural key for behavioural features.

In [7]:
prof = ['first','last','street','city','state','zip','dob','job']
both = pd.concat([train, test])
print('max distinct profile values per card:', both.groupby('cc_num')[prof].nunique().max().max())
print('cards:', both.cc_num.nunique())

max distinct profile values per card: 1
cards: 999


## The fraud-only-card artefact
Cards whose transactions are *all* fraud, and cards that appear only in test.

In [8]:
g = both.groupby(['cc_num'])[config.TARGET].agg(['mean','size'])
new = set(test.cc_num) - set(train.cc_num)
print('cards that are 100% fraud:', int((g['mean'] == 1).sum()))
print('test-only cards:', len(new), '| their fraud share:', g.loc[list(new), 'mean'].mean())

cards that are 100% fraud: 91
test-only cards: 16 | their fraud share: 1.0


Full write-ups: `reports/data_inspection.md`, `reports/data_inspection_notes.md`, `reports/data_leakage_analysis.md`.